# Project: REINFORCE plays CartPole 🎮
Train a **neural-network policy** to balance a pole using the **policy gradient** theorem: make actions that led to high returns more likely. No value table, no replay buffer: play an episode, compute discounted returns, nudge $\log\pi(a|s)$ up in proportion. It's the ancestor of PPO, the algorithm behind RLHF.

Topic: [[Q-Learning and Policy Gradients]], [[RL Basics and MDPs]] · guide note: [[Project - REINFORCE Plays CartPole]] · compare: [[Fun Projects]] → CartPole Neuroevolution (no gradients)

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import math, numpy as np, matplotlib.pyplot as plt, torch, torch.nn as nn
torch.manual_seed(0)
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
G, M_CART, M_POLE, L, F_MAG, TAU = 9.8, 1.0, 0.1, 0.5, 10.0, 0.02          # classic CartPole constants (Barto, Sutton & Anderson 1983)
class CartPole:
    def reset(self, rng):
        self.s = rng.uniform(-0.05, 0.05, 4); self.t = 0; return self.s.copy()
    def step(self, action):
        x, x_dot, th, th_dot = self.s; force = F_MAG if action == 1 else -F_MAG
        total = M_CART + M_POLE; tmp = (force + M_POLE * L * th_dot ** 2 * math.sin(th)) / total
        th_acc = (G * math.sin(th) - math.cos(th) * tmp) / (L * (4 / 3 - M_POLE * math.cos(th) ** 2 / total))
        x_acc = tmp - M_POLE * L * th_acc * math.cos(th) / total
        self.s = np.array([x + TAU * x_dot, x_dot + TAU * x_acc, th + TAU * th_dot, th_dot + TAU * th_acc]); self.t += 1
        done = abs(self.s[0]) > 2.4 or abs(self.s[2]) > 12 * math.pi / 180 or self.t >= 500
        return self.s.copy(), 1.0, done

## 1. Discounted returns\n$G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \dots$, computed backwards in one pass. Optionally **normalise** (subtract mean, divide by std): a simple baseline that reduces variance a lot. Return a float tensor.

In [ ]:
def discounted_returns(rewards, gamma=0.99, normalize=True):
    # TODO 1: backward pass G = r + γG, then (optionally) standardise
    raise NotImplementedError  # TODO 1

## 2. The policy-gradient loss\n$\nabla J(\theta) = \mathbb E\big[\sum_t \nabla\log\pi_\theta(a_t|s_t)\,G_t\big]$. As a loss to *minimise*: $-\sum_t \log\pi_\theta(a_t|s_t)\,G_t$.

In [ ]:
def pg_loss(log_probs, returns):
    # TODO 2: negative sum of log-probabilities weighted by returns
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_pg():
    r = discounted_returns([1, 1, 1], gamma=0.5, normalize=False)
    lp = torch.tensor([-0.1, -0.2], requires_grad=True); l = pg_loss(lp, torch.tensor([2.0, -1.0])); l.backward()
    return torch.allclose(r, torch.tensor([1.75, 1.5, 1.0])) and abs(discounted_returns([1.0] * 5).mean()) < 1e-6 and torch.allclose(lp.grad, torch.tensor([-2.0, 1.0]))
check("policy gradient", _t_pg, "Returns of [1,1,1] with γ=0.5 are [1.75, 1.5, 1]; the loss gradient w.r.t. log π must be −G.")
history = []
if ready("policy gradient"):
    rng = np.random.default_rng(0); env = CartPole()
    policy = nn.Sequential(nn.Linear(4, 32), nn.Tanh(), nn.Linear(32, 2)); opt = torch.optim.Adam(policy.parameters(), lr=0.01)
    for episode in range(400):
        s, done, log_probs, rewards = env.reset(rng), False, [], []
        while not done:
            dist = torch.distributions.Categorical(logits=policy(torch.tensor(s, dtype=torch.float32)))
            a = dist.sample(); log_probs.append(dist.log_prob(a)); s, r, done = env.step(int(a)); rewards.append(r)
        loss = pg_loss(torch.stack(log_probs), discounted_returns(rewards)); opt.zero_grad(); loss.backward(); opt.step()
        history.append(len(rewards))
        if np.mean(history[-20:]) >= 475: break
    plt.plot(history, alpha=.5); plt.plot(np.convolve(history, np.ones(20) / 20, "valid"), lw=2); plt.xlabel("episode"); plt.ylabel("steps balanced"); plt.show()
    print(f"episodes: {len(history)}; mean of the last 20: {np.mean(history[-20:]):.0f} steps (500 = perfect)")
    check("learns to balance", lambda: np.mean(history[-20:]) >= 400, "The last 20 episodes should average ≥ 400 steps.")

<details><summary>▶ Solution</summary>

```python
def discounted_returns(rewards, gamma=0.99, normalize=True):
    G, out = 0.0, []
    for r in reversed(rewards):
        G = r + gamma * G
        out.append(G)
    out = torch.tensor(out[::-1], dtype=torch.float32)
    return (out - out.mean()) / (out.std() + 1e-8) if normalize else out
```

```python
def pg_loss(log_probs, returns):
    return -(log_probs * returns).sum()
```
</details>

## Stretch goals
- Turn off normalisation (`normalize=False`). How much slower and noisier does learning get?
- Add a learned value baseline (actor–critic): subtract $V(s_t)$ instead of the mean return.
- Run 5 seeds and plot mean ± std. RL results are notoriously seed-dependent.